# Sprint 4: Three-settlement extension (Mathare + Mukuru)

Proposal weeks 7–8: retrain / fine-tune **both** model heads on Kibera + Mathare + Mukuru and report whether the validation bars are met.

| Head | Metric | Target |
|------|--------|--------|
| Attention U-Net (flood) | IoU (transfer bar) / F1 (NFR) | ≥ 0.60 / ≥ 0.75 |
| GraphSAGE (deprivation) | macro-F1 | ≥ 0.70 (`config.GRAPHSAGE_FINAL_MACRO_F1_TARGET`) |

**What this notebook does**

1. Checks out the Sprint 4 branch (or `main` once merged).
2. Rebuilds proximity graphs for all three settlements (Overpass mirror retries).
3. Loads tiled flood stacks when present on Drive; scaffolds the path otherwise.
4. Jointly trains GraphSAGE on the combined three-settlement graph (per-settlement stratified split).
5. Trains / fine-tunes the Attention U-Net on pooled tiles with per-settlement spatial holdout (no cross-settlement location leakage).
6. Prints per-settlement + pooled metrics and whether thresholds are met.

**Offline / Cloud Agent note.** Real Mathare/Mukuru GEE exports and Overpass pulls only run here in Colab. The `src/models/multi_settlement.py` helpers are covered by offline unit tests with synthetic data.


## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** recommended (U-Net fine-tune benefits; GraphSAGE is fine on CPU).
- Sprint 1 tiling for each settlement should produce `data/tiles/<settlement>/manifest_train.json` on Drive (or rebuild graphs-only first and tile later).
- Set `GEE_PROJECT_ID` if you will kick off missing raster exports.
- Mount Drive so graphs, tiles, and checkpoints persist across Colab disconnects.


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — GraphSAGE is fine on CPU; U-Net will be slower.")


CUDA available: True
GPU: Tesla T4


In [5]:
%%bash
# Prefer the Sprint 4 PR branch while open; switch BRANCH to main after merge.
BRANCH=cursor/sprint4-three-settlements-cc98
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git

if [ -d /content/nairobi-risk-mapping/.git ]; then
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || git fetch origin main
  git checkout "$BRANCH" 2>/dev/null || git checkout main
  git pull --ff-only origin "$BRANCH" 2>/dev/null || git pull --ff-only origin main
else
  git clone "$REPO" /content/nairobi-risk-mapping
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || true
  git checkout "$BRANCH" 2>/dev/null || git checkout main
fi
cd /content/nairobi-risk-mapping
git rev-parse --abbrev-ref HEAD
ls src/models/multi_settlement.py


Your branch is up to date with 'origin/cursor/sprint4-three-settlements-cc98'.
Already up to date.
cursor/sprint4-three-settlements-cc98
src/models/multi_settlement.py


From https://github.com/emmanuelamor5/nairobi-risk-mapping
 * branch            cursor/sprint4-three-settlements-cc98 -> FETCH_HEAD


In [6]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt


/content/nairobi-risk-mapping


In [7]:
# --- config knobs ---
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    # Persist heavy artefacts on Drive (Colab local disk is wiped on disconnect).
    DRIVE_ROOT = Path("/content/drive/MyDrive/nairobi_risk_mapping")
except ImportError:
    print("Not in Colab — artefacts stay under the repo working directory.")
    DRIVE_ROOT = Path(".")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Symlink repo data/ -> Drive so DataPipeline paths stay relative.
DATA_LINK = Path("data")
if not (DATA_LINK.is_symlink() or DATA_LINK.exists()):
    target = DRIVE_ROOT / "data"
    target.mkdir(parents=True, exist_ok=True)
    DATA_LINK.symlink_to(target)

SETTLEMENT_KEYS = ("kibera", "mathare", "mukuru")
REBUILD_GRAPHS = False          # True forces fresh OSM/Overpass pulls
INCLUDE_RASTER_EXPORT = False  # True kicks off async GEE exports (slow)
FINE_TUNE_UNET_FROM = None     # e.g. "checkpoints/kibera_attention_unet_best.pt"
GEE_PROJECT_ID = None          # or os.environ / your GCP project id
GRAPHSAGE_EPOCHS = 100
UNET_EPOCHS = 40
CHECKPOINT_DIR = str(DRIVE_ROOT / "checkpoints" / "sprint4")
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("DATA ->", DATA_LINK.resolve())
print("CHECKPOINTS ->", CHECKPOINT_DIR)

gs_threshold = None
unet_threshold = None
unet_result = None


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA -> /content/drive/MyDrive/nairobi_risk_mapping/data
CHECKPOINTS -> /content/drive/MyDrive/nairobi_risk_mapping/checkpoints/sprint4


## 1. Repo check — Sprint 4 APIs exist


In [8]:
from src import config
from src.models.multi_settlement import (
    combine_settlement_graphs,
    evaluate_flood_entries_by_settlement,
    evaluate_graphsage_by_settlement,
    flood_threshold_report,
    graphsage_threshold_report,
    load_multi_settlement_manifests,
    load_settlement_graphs,
    multi_settlement_spatial_holdout,
    prepare_multi_settlement_data,
)
from src.models.graphsage_model import GraphSAGEModel
from src.models.attention_unet_model import AttentionUNetModel
from src.models.dataset import EVENT_S1_LAYOUT, DEFAULT_LAYOUT

assert config.GRAPHSAGE_FINAL_MACRO_F1_TARGET == 0.70
assert config.ATTENTION_UNET_TRANSFER_IOU_TARGET == 0.60
assert config.ATTENTION_UNET_FINAL_F1_TARGET == 0.75
assert tuple(config.SETTLEMENT_KEYS) == SETTLEMENT_KEYS
print("Sprint 4 targets OK:", {
    "graphsage_macro_f1": config.GRAPHSAGE_FINAL_MACRO_F1_TARGET,
    "unet_iou": config.ATTENTION_UNET_TRANSFER_IOU_TARGET,
    "unet_f1": config.ATTENTION_UNET_FINAL_F1_TARGET,
})


Sprint 4 targets OK: {'graphsage_macro_f1': 0.7, 'unet_iou': 0.6, 'unet_f1': 0.75}


## 2. Rebuild / load graphs for all three settlements

Uses `DataPipeline.acquire_graph_data()` (Overpass mirror retries from Sprint 3). Set `REBUILD_GRAPHS=True` if Drive graphs predate deprivation labels.


In [9]:
from src.data_pipeline import DataPipeline
import torch

graphs_dir = Path("data/graphs")
graphs_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_GRAPHS or not all((graphs_dir / f"{k}_graph.pt").exists() for k in SETTLEMENT_KEYS):
    for key in SETTLEMENT_KEYS:
        print("=" * 60, key)
        pipe = DataPipeline(settlement_key=key, gee_project_id=GEE_PROJECT_ID)
        if INCLUDE_RASTER_EXPORT:
            task = pipe.acquire_raster_stack()
            print("GEE export task:", task)
        data, integrity = pipe.acquire_graph_data()
        print(key, "nodes=", data.num_nodes, "integrity=", integrity)
else:
    print("Using existing graphs in", graphs_dir.resolve())

graphs = load_settlement_graphs(graphs_dir, settlement_keys=SETTLEMENT_KEYS)
print("Loaded settlements:", list(graphs))
for k, g in graphs.items():
    print(f"  {k}: nodes={g.num_nodes} y_unique={torch.unique(g.y).tolist()}")


Using existing graphs in /content/drive/MyDrive/nairobi_risk_mapping/data/graphs
Loaded settlements: ['kibera', 'mathare', 'mukuru']
  kibera: nodes=11053 y_unique=[0, 1, 2]
  mathare: nodes=5777 y_unique=[0, 1, 2]
  mukuru: nodes=1175 y_unique=[0, 1, 2]


## 3. GraphSAGE — joint three-settlement training

Graphs are combined as a **disjoint union** (no cross-settlement edges). Splits are stratified **within each settlement** (60/20/20). Selection metric: validation macro-F1.


In [10]:
if len(graphs) < 1:
    raise RuntimeError("No graphs loaded — rebuild with REBUILD_GRAPHS=True.")

combined = prepare_multi_settlement_data(graphs, seed=42)
print("Combined nodes:", combined.num_nodes)
print("Split sizes:",
      int(combined.train_mask.sum()),
      int(combined.val_mask.sum()),
      int(combined.test_mask.sum()))

gs_model = GraphSAGEModel(
    in_channels=combined.x.size(1),
    hidden_channels=64,
    num_classes=config.DEPRIVATION_NUM_CLASSES,
    dropout=0.5,
)
gs_result = gs_model.fit(
    combined,
    epochs=GRAPHSAGE_EPOCHS,
    learning_rate=5e-3,
    weight_decay=5e-4,
    checkpoint_dir=str(Path(CHECKPOINT_DIR) / "graphsage"),
    early_stopping_patience=25,
    seed=42,
)
print("best_val_macro_f1:", gs_result["best_val_macro_f1"])
print("meets_final_target (pooled val):", gs_result["meets_final_target"],
      "target=", gs_result["final_target"])


Combined nodes: 18005
Split sizes: 10803 3601 3601
best_val_macro_f1: 0.8812804660795578
meets_final_target (pooled val): True target= 0.7


In [11]:
# Per-settlement + pooled reports on val and test
for split in ("val", "test"):
    report = gs_model.evaluate_by_settlement(combined, split=split)
    print(f"\n=== GraphSAGE {split} ===")
    for key, r in report["per_settlement"].items():
        print(f"  {key:8s}  macro_f1={r['macro_f1']:.4f}  acc={r['accuracy']:.4f}")
    print(f"  {'pooled':8s}  macro_f1={report['pooled']['macro_f1']:.4f}")
    thr = graphsage_threshold_report(
        {k: v["macro_f1"] for k, v in report["per_settlement"].items()},
        report["pooled"]["macro_f1"],
    )
    print("  threshold:", {k: thr[k] for k in (
        "meets_pooled_target", "all_settlements_meet_target", "target",
    )})
    if split == "val":
        gs_threshold = thr



=== GraphSAGE val ===
  kibera    macro_f1=0.8972  acc=0.8955
  mathare   macro_f1=0.8755  acc=0.8753
  mukuru    macro_f1=0.7453  acc=0.7574
  pooled    macro_f1=0.8813
  threshold: {'meets_pooled_target': True, 'all_settlements_meet_target': True, 'target': 0.7}

=== GraphSAGE test ===
  kibera    macro_f1=0.9011  acc=0.9000
  mathare   macro_f1=0.8930  acc=0.8927
  mukuru    macro_f1=0.7703  acc=0.7745
  pooled    macro_f1=0.8906
  threshold: {'meets_pooled_target': True, 'all_settlements_meet_target': True, 'target': 0.7}


## 4. Attention U-Net — pooled tiles + per-settlement spatial holdout

Requires Sprint 1 tiled stacks on Drive (`data/tiles/<settlement>/manifest_train.json`). If Mathare/Mukuru tiles are missing, this section skips with a clear message — rebuild via GEE export + `tile_raster_stack` in notebook 01 / 02 v2 first.

Holdouts are **per settlement**: augmented variants of the same `(row_off, col_off)` stay together; the same numeric offsets in different settlements are independent.

> **Note — Kibera tile count (8 → 16) vs component pruning.** Kibera's tile count rose from 8 to 16 when the size-aware export buffer replaced the fixed 200 m buffer: the larger raster fits a 2×2 window grid (4 distinct spatial locations) instead of 1×2 (2 locations), and each location yields 4 train augmentations (4 × 4 = 16 vs 2 × 4 = 8). This is purely a **raster-path** effect — tile count depends only on the exported GeoTIFF's width/height, the 60% valid-pixel filter, and augmentation. It is **independent of the connected-component pruning** applied in the **graph path** (`acquire_graph_data` → GraphSAGE), which drops nodes outside the largest connected component and only changes `data.num_nodes` / `y` for the deprivation head. The two heads share nothing at data-prep level except the settlement boundary, so graph pruning never affects the U-Net tiles.


In [12]:
!cd /content/nairobi-risk-mapping && git fetch origin && git checkout cursor/sprint4-three-settlements-cc98 && git pull origin cursor/sprint4-three-settlements-cc98
!cd /content/nairobi-risk-mapping && git log --oneline -3

Already on 'cursor/sprint4-three-settlements-cc98'
Your branch is up to date with 'origin/cursor/sprint4-three-settlements-cc98'.
From https://github.com/emmanuelamor5/nairobi-risk-mapping
 * branch            cursor/sprint4-three-settlements-cc98 -> FETCH_HEAD
Already up to date.
a07c7a8 (HEAD -> cursor/sprint4-three-settlements-cc98, origin/cursor/sprint4-three-settlements-cc98) Size-aware raster export buffer so small settlements tile
625fedd Handle empty / under-populated tile manifests in Sprint 4 split
e2b3c99 Add Sprint 4 Colab notebook and README status for three settlements.


In [16]:
import inspect
from src.data_pipeline import DataPipeline

src = inspect.getsource(DataPipeline.acquire_raster_stack)
assert "required_export_buffer_m" in src, "STALE module — pull a07c7a8 then restart again"
assert "geometry.buffer(200)" not in src, "STALE module — old fixed buffer still present"
print("OK: size-aware export buffer is active")

OK: size-aware export buffer is active


In [17]:
import geopandas as gpd
from src import config
from src.utils.graph_utils import fetch_settlement_boundary
from src.utils.gee_utils import required_export_buffer_m

for key in ("mathare", "mukuru"):
    b = fetch_settlement_boundary(key).to_crs(config.CRS_METRIC).total_bounds
    print(key, "extent_m", round(b[2]-b[0]), "x", round(b[3]-b[1]),
          "-> buffer", round(required_export_buffer_m(b)), "m")

mathare extent_m 4466 x 1982 -> buffer 1550 m
mukuru extent_m 1161 x 544 -> buffer 2269 m


In [18]:
import time, rasterio
from pathlib import Path
from src.data_pipeline import DataPipeline
from src.models.dataset import EVENT_S1_LAYOUT

GEE_PROJECT_ID = "mtl-nairobi-2026"
DRIVE = "/content/drive/MyDrive/nairobi_risk_mapping"

tasks = {}
for key in ("mathare", "mukuru"):
    pipe = DataPipeline(settlement_key=key, gee_project_id=GEE_PROJECT_ID)
    tasks[key] = pipe.acquire_raster_stack(label_mode="event", include_s1_inputs=True)

while True:
    st = {k: t.status().get("state") for k, t in tasks.items()}
    print(st)
    if all(s in ("COMPLETED", "FAILED", "CANCELLED") for s in st.values()):
        break
    time.sleep(60)

for key in ("mathare", "mukuru"):
    stack_path = f"{DRIVE}/{key}_raster_stack.tif"
    with rasterio.open(stack_path) as src:
        print(key, "raster size:", src.width, "x", src.height)
    pipe = DataPipeline(settlement_key=key, gee_project_id=GEE_PROJECT_ID)
    print(key, "->", pipe.tile_raster_stack(
        stack_path, split="train", band_names=list(EVENT_S1_LAYOUT.band_names)))

{'mathare': 'READY', 'mukuru': 'READY'}
{'mathare': 'COMPLETED', 'mukuru': 'RUNNING'}
{'mathare': 'COMPLETED', 'mukuru': 'RUNNING'}
{'mathare': 'COMPLETED', 'mukuru': 'COMPLETED'}
mathare raster size: 758 x 511
mathare -> data/tiles/mathare/manifest_train.json
mukuru raster size: 571 x 510
mukuru -> data/tiles/mukuru/manifest_train.json


In [21]:
import time, glob, os, rasterio
from src.data_pipeline import DataPipeline
from src.models.dataset import EVENT_S1_LAYOUT

GEE_PROJECT_ID = "mtl-nairobi-2026"
DRIVE = "/content/drive/MyDrive/nairobi_risk_mapping"

# 1) clear Kibera's old 10-band stack (base + suffixed dups) and its old tiles
for f in sorted(glob.glob(f"{DRIVE}/kibera_raster_stack*.tif")):
    print("deleting", f); os.remove(f)
for f in sorted(glob.glob("data/tiles/kibera/*")):
    os.remove(f)

# 2) re-export Kibera WITH S1 inputs (12-band EVENT_S1, like Mathare/Mukuru)
pipe = DataPipeline(settlement_key="kibera", gee_project_id=GEE_PROJECT_ID)
task = pipe.acquire_raster_stack(label_mode="event", include_s1_inputs=True)
while task.status().get("state") not in ("COMPLETED", "FAILED", "CANCELLED"):
    print(task.status().get("state")); time.sleep(60)
print(task.status().get("state"))

# 3) tile it (fixed dB window for the VV bands)
stack_path = sorted(glob.glob(f"{DRIVE}/kibera_raster_stack*.tif"))[-1]
with rasterio.open(stack_path) as src:
    print("kibera raster size:", src.width, "x", src.height, "bands", src.count)
print("kibera ->", pipe.tile_raster_stack(
    stack_path, split="train", band_names=list(EVENT_S1_LAYOUT.band_names)))

deleting /content/drive/MyDrive/nairobi_risk_mapping/kibera_raster_stack (1).tif
deleting /content/drive/MyDrive/nairobi_risk_mapping/kibera_raster_stack (2).tif
deleting /content/drive/MyDrive/nairobi_risk_mapping/kibera_raster_stack (3).tif
deleting /content/drive/MyDrive/nairobi_risk_mapping/kibera_raster_stack (4).tif
deleting /content/drive/MyDrive/nairobi_risk_mapping/kibera_raster_stack.tif


/content/nairobi-risk-mapping/src/utils/graph_utils.py:88: UserWarning: OSM geocoding failed or implausible for 'Kibera, Nairobi, Kenya' (Nominatim did not geocode query 'Kibera, Nairobi, Kenya' to a geometry of type (Multi)Polygon.); falling back to hardcoded bbox. VERIFY this boundary visually before using it for training -- see config.py docstring.
  warnings.warn(


READY
RUNNING
COMPLETED
kibera raster size: 578 x 511 bands 12
kibera -> data/tiles/kibera/manifest_train.json


In [22]:
import numpy as np

tiles_root = Path("data/tiles")
manifests = load_multi_settlement_manifests(
    tiles_root, settlement_keys=SETTLEMENT_KEYS, split="train", require_all=False,
)
print("Tile manifests found:", {k: len(v) for k, v in manifests.items()})

unet_threshold = None
unet_result = None

if len(manifests) < 2:
    print(
        "SKIPPING U-Net training: need tile manifests for at least two settlements. "
        "Run Sprint 1 tiling for Mathare/Mukuru in Colab (GEE → Drive → tile_raster_stack), "
        "then re-run this section. GraphSAGE above does not depend on tiles."
    )
else:
    split = multi_settlement_spatial_holdout(manifests, holdout_fraction=0.25, seed=42)
    print("Holdout locations:", split["holdout_locations"])
    print("Train/val entries:", len(split["train"]), len(split["val"]))

    # Prefer EVENT_S1_LAYOUT when tile band count matches (Sprint 2 v2 stacks).
    sample = np.load(split["train"][0]["path"])
    layout = EVENT_S1_LAYOUT if sample.shape[0] == len(EVENT_S1_LAYOUT.band_names) else DEFAULT_LAYOUT
    print("Using layout:", layout.band_names, "in_channels=", layout.in_channels)

    unet = AttentionUNetModel(
        layout=layout,
        dropout=0.3,
        pretrained_checkpoint=None,  # set Sen1Floods11 path if available
        freeze_pretrained_encoder=False,
    )
    if FINE_TUNE_UNET_FROM:
        unet.load_checkpoint(FINE_TUNE_UNET_FROM)
        print("Fine-tuning from", FINE_TUNE_UNET_FROM)

    unet_result = unet.fit(
        split["train"],
        split["val"],
        epochs=UNET_EPOCHS,
        batch_size=4,
        learning_rate=1e-3,
        weight_decay=1e-4,
        early_stopping_patience=12,
        checkpoint_dir=str(Path(CHECKPOINT_DIR) / "attention_unet"),
    )
    print("best_val_iou:", unet_result["best_val_iou"])

    # Reload best checkpoint for settlement-wise summary on the val set.
    best_ckpt = Path(CHECKPOINT_DIR) / "attention_unet" / "attention_unet_best.pt"
    if best_ckpt.exists():
        unet.load_checkpoint(str(best_ckpt))

    flood_eval = evaluate_flood_entries_by_settlement(unet, split["val"])
    print("\n=== U-Net val (per settlement + pooled) ===")
    for key, r in flood_eval["per_settlement"].items():
        print(f"  {key:8s}  iou={r['iou']:.4f}  f1={r['f1']:.4f}")
    print(f"  {'pooled':8s}  iou={flood_eval['pooled']['iou']:.4f}  "
          f"f1={flood_eval['pooled']['f1']:.4f}")

    unet_threshold = flood_threshold_report(
        {k: {"iou": v["iou"], "f1": v["f1"]} for k, v in flood_eval["per_settlement"].items()},
        flood_eval["pooled"]["iou"],
        flood_eval["pooled"]["f1"],
    )
    print("threshold:", {
        k: unet_threshold[k] for k in (
            "meets_pooled_iou_target", "meets_pooled_f1_target",
            "iou_target", "f1_target",
        )
    })


Tile manifests found: {'kibera': 16, 'mathare': 20, 'mukuru': 16}
Holdout locations: {'kibera': [(206, 206)], 'mathare': [(206, 206)], 'mukuru': [(206, 206)]}
Train/val entries: 40 12
Using layout: ('B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'pre_vv', 'event_vv', 'flood_label', 'elevation', 'slope', 'twi') in_channels= 11
best_val_iou: 0.7204604645576954

=== U-Net val (per settlement + pooled) ===
  kibera    iou=0.7360  f1=0.8479
  mathare   iou=0.6379  f1=0.7789
  mukuru    iou=0.7608  f1=0.8642
  pooled    iou=0.7201  f1=0.8373
threshold: {'meets_pooled_iou_target': True, 'meets_pooled_f1_target': True, 'iou_target': 0.6, 'f1_target': 0.75}


## 5. Threshold summary


In [23]:
print("=" * 60)
print("SPRINT 4 THRESHOLD SUMMARY")
print("=" * 60)

print("\nGraphSAGE (deprivation) — target macro-F1 >=",
      config.GRAPHSAGE_FINAL_MACRO_F1_TARGET)
if gs_threshold is not None:
    print("  pooled meets target:", gs_threshold["meets_pooled_target"],
          f"(macro_f1={gs_threshold['pooled_macro_f1']:.4f})")
    for k, v in gs_threshold["per_settlement_macro_f1"].items():
        print(f"  {k:8s} macro_f1={v:.4f}  meets={gs_threshold['meets_per_settlement_target'][k]}")
    print("  all settlements meet:", gs_threshold["all_settlements_meet_target"])
else:
    print("  (no GraphSAGE threshold report — training cell did not run)")

print("\nAttention U-Net (flood) — IoU >=",
      config.ATTENTION_UNET_TRANSFER_IOU_TARGET,
      "/ F1 >=", config.ATTENTION_UNET_FINAL_F1_TARGET)
if unet_threshold is not None:
    print("  pooled IoU meets:", unet_threshold["meets_pooled_iou_target"],
          f"(iou={unet_threshold['pooled_iou']:.4f})")
    print("  pooled F1 meets:", unet_threshold["meets_pooled_f1_target"],
          f"(f1={unet_threshold['pooled_f1']:.4f})")
    for k in unet_threshold["per_settlement_iou"]:
        print(f"  {k:8s} iou={unet_threshold['per_settlement_iou'][k]:.4f}  "
              f"f1={unet_threshold['per_settlement_f1'][k]:.4f}")
else:
    print("  (U-Net skipped — tile manifests missing for Mathare/Mukuru; run tiling in Colab)")

print("\nCheckpoints under:", CHECKPOINT_DIR)
print("Next: Sprint 5 late-fusion policy engine (do not start here).")


SPRINT 4 THRESHOLD SUMMARY

GraphSAGE (deprivation) — target macro-F1 >= 0.7
  pooled meets target: True (macro_f1=0.8813)
  kibera   macro_f1=0.8972  meets=True
  mathare  macro_f1=0.8755  meets=True
  mukuru   macro_f1=0.7453  meets=True
  all settlements meet: True

Attention U-Net (flood) — IoU >= 0.6 / F1 >= 0.75
  pooled IoU meets: True (iou=0.7201)
  pooled F1 meets: True (f1=0.8373)
  kibera   iou=0.7360  f1=0.8479
  mathare  iou=0.6379  f1=0.7789
  mukuru   iou=0.7608  f1=0.8642

Checkpoints under: /content/drive/MyDrive/nairobi_risk_mapping/checkpoints/sprint4
Next: Sprint 5 late-fusion policy engine (do not start here).


## Done

- GraphSAGE joint training path is fully wired offline + in this notebook.
- U-Net multi-settlement path needs real Mathare/Mukuru tiles from GEE (Colab-only).
- Do **not** treat synthetic Cloud-Agent metrics as real accuracy — re-run this notebook in Colab for reportable numbers.
